# SegOCR — Train Workers 1 + 2 in Parallel (Kaggle, 2× T4)

This notebook trains **workers 1 and 2** of the 5-worker ensemble simultaneously — each pinned to one of Kaggle's two T4 GPUs as an independent training process. No DataParallel, no gradient sync. Both run at full single-GPU speed in parallel.

**One Kaggle session → two trained workers.** This is the efficient way to use Kaggle's 2× T4 for an ensemble (vs DataParallel, which actually runs *slower* than single-GPU for this model size).

**Before running:**
1. Settings → Accelerator: **GPU T4 x2** (must be the two-GPU option).
2. Add Data (sidebar) → attach **both** `segocr-ensemble-a` and `segocr-ensemble-b`.
3. (Optional, for resume) Add Data → Notebook Output Files → attach your previous version's output of this notebook.
4. Click **Save Version → Save & Run All** (runs headless for ~8 hr).

When done, you'll have:
- `/kaggle/working/weights_w1/averaged_best.pth`
- `/kaggle/working/weights_w2/averaged_best.pth`
Both appear as downloadable notebook outputs.

## 1 / Setup — clone repo + install deps

In [ ]:
import os
if not os.path.isdir('/kaggle/working/segocr'):
    !git clone https://github.com/mauryantitans/SegOCR.git /kaggle/working/segocr
%cd /kaggle/working/segocr
!git pull --quiet
!pip install -q -e .
!pip install -q -r requirements/base.txt
!pip install -q segmentation-models-pytorch

## 2 / Verify 2 GPUs visible

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA. Settings → Accelerator → GPU T4 x2.')
n_gpus = torch.cuda.device_count()
print(f'CUDA devices visible: {n_gpus}')
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    mem_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  cuda:{i}  {name}  ({mem_gb:.1f} GB)')
if n_gpus < 2:
    raise RuntimeError(
        f'Need 2 GPUs for this notebook (got {n_gpus}). '
        'Settings → Accelerator → GPU T4 x2.'
    )

## 3 / Worker config (locked: workers 1 + 2)

Both workers attach the same two datasets but use disjoint slices. Each worker gets its own weights dir and seed.

In [ ]:
WORKER_A = 1    # trained on cuda:0
WORKER_B = 2    # trained on cuda:1
WORKERS = [WORKER_A, WORKER_B]

import os, glob, shutil
DATASET_SLUG_A = 'segocr-ensemble-a'   # indices 0..39999
DATASET_SLUG_B = 'segocr-ensemble-b'   # indices 40000..79999

def find_dataset_worker_dir(slug, worker_id):
    '''Locate /kaggle/input/.../worker{N} for a given dataset slug.'''
    patterns = [
        f'/kaggle/input/{slug}/worker{worker_id}',
        f'/kaggle/input/{slug}/{slug}/worker{worker_id}',
        f'/kaggle/input/datasets/*/{slug}/worker{worker_id}',
        f'/kaggle/input/datasets/*/{slug}/{slug}/worker{worker_id}',
    ]
    for pat in patterns:
        matches = sorted(glob.glob(pat))
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f'Could not find worker{worker_id} in dataset {slug!r}. '
        f'Tried: {patterns}. Did you attach the dataset via Add Data?'
    )

DATA_DIRS = {}
WEIGHTS_DIRS = {}
for w in WORKERS:
    a_dir = find_dataset_worker_dir(DATASET_SLUG_A, w)
    b_dir = find_dataset_worker_dir(DATASET_SLUG_B, w)
    merged = f'/kaggle/working/segocr_data_w{w}_merged'
    if os.path.isdir(merged):
        shutil.rmtree(merged)
    for subdir in ('images', 'semantic', 'instance', 'metadata'):
        os.makedirs(f'{merged}/{subdir}', exist_ok=True)
        for src_base in (a_dir, b_dir):
            src_dir = f'{src_base}/{subdir}'
            if not os.path.isdir(src_dir):
                continue
            for fname in os.listdir(src_dir):
                src = f'{src_dir}/{fname}'
                dst = f'{merged}/{subdir}/{fname}'
                if not os.path.exists(dst):
                    os.symlink(src, dst)
    DATA_DIRS[w] = merged
    WEIGHTS_DIRS[w] = f'/kaggle/working/weights_w{w}'
    os.makedirs(WEIGHTS_DIRS[w], exist_ok=True)
    n_merged = len(os.listdir(f'{merged}/images'))
    print(f'Worker {w}: data={merged} ({n_merged} imgs), weights={WEIGHTS_DIRS[w]}')

## 4 / Resume from previous version's output (if attached)

If a previous run of this notebook was saved and re-attached via **Add Data → Notebook Output Files**, its checkpoints get auto-discovered and copied per worker.

In [ ]:
import shutil, glob
for w in WORKERS:
    prev = []
    for pat in (
        f'/kaggle/input/*/weights_w{w}/checkpoint_*.pth',
        f'/kaggle/input/*/weights_w{w}/averaged_best.pth',
        f'/kaggle/input/*/weights_w{w}/snapshot_*.pth',
    ):
        prev.extend(glob.glob(pat))
    if prev:
        print(f'Worker {w}: found {len(prev)} previous checkpoint(s); copying...')
        for src in prev:
            dst = os.path.join(WEIGHTS_DIRS[w], os.path.basename(src))
            shutil.copy(src, dst)
            print(f'  {src} -> {dst}')
    else:
        print(f'Worker {w}: no previous checkpoints; starting fresh.')

## 5 / Build per-worker training configs

Same hyperparameters as the single-worker notebooks (batch=16, 30K iters, 512²). Only data_dir + output_dir differ between the two configs.

In [ ]:
import yaml
from pathlib import Path

IMG_SIZE = 512
TOTAL_ITERS = 30_000
BATCH_SIZE = 16

def build_cfg_for_worker(w):
    cfg = yaml.safe_load(Path('segocr/configs/default.yaml').read_text())
    cfg['generator']['fonts']['root_dir'] = '/usr/share/fonts'
    cfg['generator']['fonts']['cache_path'] = '/kaggle/working/font_cache.json'
    cfg['generator']['fonts']['min_size'] = 40
    cfg['generator']['fonts']['max_size'] = 128
    cfg['generator']['image_size'] = [IMG_SIZE, IMG_SIZE]
    cfg['generator']['num_workers'] = 2
    cfg['generator']['output_dir'] = DATA_DIRS[w]
    cfg['generator']['text']['min_length'] = 2
    cfg['generator']['text']['max_length'] = 20
    cfg['generator']['text']['min_words_per_line'] = 1
    cfg['generator']['text']['max_words_per_line'] = 3
    cfg['generator']['text']['max_lines'] = 1
    cfg['generator']['text']['case_distribution'] = {
        'lower': 0.50, 'upper': 0.20, 'mixed': 0.20, 'title': 0.10,
    }
    cfg['generator']['text']['rare_char_boost'] = 4.0
    cfg['generator']['text']['corpus_paths'] = [
        {'path': 'BUNDLED:signs', 'tag': 'signs', 'weight': 0.30},
        {'path': 'BUNDLED:receipts', 'tag': 'receipts', 'weight': 0.20},
        {'path': 'BUNDLED:names', 'tag': 'names', 'weight': 0.20},
        {'path': 'BUNDLED:numbers', 'tag': 'numbers', 'weight': 0.30},
    ]
    cfg['generator']['layout']['modes'] = {
        'horizontal': 0.50, 'rotated': 0.20, 'curved': 0.10,
        'perspective': 0.10, 'deformed': 0.10, 'paragraph': 0.0,
    }
    cfg['generator']['background']['natural_image_dirs'] = []
    cfg['generator']['background']['tier_distribution'] = {
        'tier1_solid': 0.40, 'tier2_procedural': 0.30,
        'tier3_natural': 0.25, 'tier4_adversarial': 0.05,
    }
    cfg['generator']['compositing']['color_strategy'] = {
        'contrast_aware': 0.60, 'random': 0.30, 'low_contrast': 0.10,
    }
    cfg['generator']['degradation']['blur'] = {
        'probability': 0.30, 'gaussian_sigma': [0.3, 1.0],
        'motion_kernel': [3, 7], 'defocus_radius': [1, 3],
    }
    cfg['generator']['degradation']['noise']['probability'] = 0.40
    cfg['generator']['degradation']['noise']['gaussian_sigma'] = [5, 20]
    cfg['generator']['degradation']['occlusion']['probability'] = 0.05
    cfg['model']['architecture'] = 'unet'
    cfg['model']['encoder'] = 'resnet18'
    cfg['model']['encoder_weights'] = 'imagenet'
    cfg['model']['head_features'] = 64
    cfg['model']['decoder_channels'] = [256, 128, 64, 32, 32]
    cfg['model']['heads'] = {'semantic': True, 'affinity': True, 'direction': True}
    cfg['model']['num_classes'] = 63
    cfg['model']['input_size'] = [IMG_SIZE, IMG_SIZE]
    cfg['training']['learning_rate'] = 3e-4
    cfg['training']['weight_decay'] = 1e-4
    cfg['training']['total_iters'] = TOTAL_ITERS
    cfg['training']['warmup_iters'] = 1_000
    cfg['training']['eval_interval'] = 2_500
    cfg['training']['save_interval'] = 2_500
    cfg['training']['log_interval'] = 100
    cfg['training']['batch_size'] = BATCH_SIZE
    cfg['training']['num_workers'] = 2
    cfg['training']['mixed_precision'] = True
    cfg['training']['output_dir'] = WEIGHTS_DIRS[w]
    cfg['training']['ema'] = {'enabled': True, 'decay': 0.999}
    cfg['training']['checkpoint_averaging'] = {'enabled': True, 'top_n': 3}
    cfg['training']['keep_best_n'] = 5
    cfg['training']['wandb'] = {'project': None}
    return cfg

CONFIG_PATHS = {}
for w in WORKERS:
    cfg = build_cfg_for_worker(w)
    path = f'/kaggle/working/train_config_w{w}.yaml'
    Path(path).write_text(yaml.safe_dump(cfg))
    CONFIG_PATHS[w] = path
    n_images = len(os.listdir(DATA_DIRS[w] + '/images'))
    print(f'Worker {w}: {n_images} samples × {TOTAL_ITERS} iters @ {IMG_SIZE}px, batch {BATCH_SIZE}')
    print(f'  config: {path}')

## 6 / Launch two parallel training processes

Each subprocess sees only one GPU (via `CUDA_VISIBLE_DEVICES`), so the training code runs as if on a single-GPU machine. They share no PyTorch state — fully independent.

Logs are written to `/kaggle/working/train_w{N}.log`. This cell polls every 5 min and prints the tail of each log so you can track progress in **Save & Run All** mode.

In [ ]:
import subprocess, sys, time

GPU_OF = {WORKER_A: 0, WORKER_B: 1}
procs = []
for w in WORKERS:
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(GPU_OF[w])
    # Each process sees just one GPU as cuda:0 -- no code changes needed.
    log_path = f'/kaggle/working/train_w{w}.log'
    log_f = open(log_path, 'w', buffering=1)
    seed = w + 1
    cmd = [
        sys.executable, '-u', '-m', 'scripts.train_model',
        '--config', CONFIG_PATHS[w],
        '--resume-latest', WEIGHTS_DIRS[w],
        '--seed', str(seed),
    ]
    proc = subprocess.Popen(cmd, env=env, stdout=log_f, stderr=subprocess.STDOUT)
    procs.append({'proc': proc, 'log': log_path, 'log_f': log_f, 'worker': w, 'gpu': GPU_OF[w]})
    print(f'Started worker {w} on cuda:{GPU_OF[w]} (PID {proc.pid}); log: {log_path}')

print(f'\nPolling every 5 min until both finish. Each worker ~7-8 hr.\n')
t0 = time.time()
while any(p['proc'].poll() is None for p in procs):
    time.sleep(300)
    elapsed_min = (time.time() - t0) / 60
    print(f'=== Status after {elapsed_min:.1f} min ===')
    for p in procs:
        rc = p['proc'].poll()
        status = 'running' if rc is None else f'exited rc={rc}'
        try:
            last = subprocess.check_output(
                ['tail', '-n', '3', p['log']], text=True
            ).strip()
        except Exception as e:
            last = f'(could not tail log: {e})'
        print(f'  worker {p["worker"]} (cuda:{p["gpu"]}): {status}')
        for line in last.splitlines():
            print(f'    | {line}')
    print()

for p in procs:
    p['log_f'].close()
    print(f"Worker {p['worker']} final exit code: {p['proc'].returncode}")
    print(f"  Full log: {p['log']}")
    print(f"  Weights:  {WEIGHTS_DIRS[p['worker']]}")

## 7 / Output files (download these)

Both `weights_wA/averaged_best.pth` and `weights_wB/averaged_best.pth` will appear in the notebook's **Output** tab. Download them along with the other workers' outputs and run `scripts/average_runs.py` locally to build the final ensemble.

In [ ]:
import os
for w in WORKERS:
    wdir = WEIGHTS_DIRS[w]
    print(f'\n/kaggle/working/weights_w{w}/:')
    if not os.path.isdir(wdir):
        print('  (missing)')
        continue
    for name in sorted(os.listdir(wdir)):
        size_mb = os.path.getsize(os.path.join(wdir, name)) / 1e6
        print(f'  {name:40s}  {size_mb:8.1f} MB')

print()
print('Next: download averaged_best.pth from each weights_wN dir,')
print('      collect all 5 worker .pth files, run scripts/average_runs.py locally.')